In [1]:
from pyspark import SparkContext
from pyspark.mllib.linalg import Vectors
import random, time

sc = SparkContext.getOrCreate()

In [2]:
def readVectorsSeq(filename):
    data = sc.textFile(filename)
    return data.map(lambda line: Vectors.dense([float(x) for x in line.split(',')])).collect()

In [3]:
def squared_distance(a, b):
    return sum((x - y) ** 2 for x, y in zip(a, b))

In [4]:
def kcenter(P, k):
    centers = [random.choice(P)]
    for _ in range(1, k):
        farthest = max(P, key=lambda p: min(squared_distance(p, c) for c in centers))
        centers.append(farthest)
    return centers

In [5]:
def kmeansPP(P, k):
    centers = [random.choice(P)]
    for _ in range(1, k):
        distances = [min(squared_distance(p, c) for c in centers) for p in P]
        total = sum(distances)
        probs = [d / total for d in distances]
        r, cum = random.random(), 0
        for i, p in enumerate(P):
            cum += probs[i]
            if r <= cum:
                centers.append(p)
                break
    return centers

In [6]:
def kmeansObj(P, C):
    return sum(min(squared_distance(p, c) for c in C) for p in P) / len(P)

In [10]:
filename = 'spam.csv'
k = 10
k1 = 20

P = readVectorsSeq(filename)
print("|------------------------------------------|")
print('| CLUSTERING OUTPUT                        |')

start = time.time()
kcenter(P, k)
print("|------------------------------------------|")
print('| k-center time:', time.time() - start)

C = kmeansPP(P, k)
print("|------------------------------------------|")
print('| kmeans++ objective:', kmeansObj(P, C))
print("|------------------------------------------|")

X = kcenter(P, k1)
C2 = kmeansPP(X, k)
print('| coreset objective:', kmeansObj(P, C2))
print("|------------------------------------------|")

|------------------------------------------|
| CLUSTERING OUTPUT                        |
|------------------------------------------|
| k-center time: 9.963270664215088
|------------------------------------------|
| kmeans++ objective: 44381.041597449774
|------------------------------------------|
| coreset objective: 333446.12326392555
|------------------------------------------|
